# Báo cáo Project
Lớp TTNT-172886, Nhóm 12

**Đề tài:** Tìm lộ trình tối ưu trên mạng tram Manchester Metrolink bằng thuật toán A* với chi phí phụ thuộc thời gian

## 1. Thông tin chung

### Thành viên
- Bùi Hải Nam 20235384
- Nguyễn Tuấn Kiệt 20235357
- Nguyễn Hà Nam 20235174

### Lịch thực hiện
- W2: Đăng ký nhóm
- W3-4: Đề xuất project (3/10)
- W08: Báo cáo tiến độ giữa kỳ (24/10)
- W15: Hoàn thành và gửi báo cáo project (12/12)
- W16-18: Trình bày project, Q&A

## 2. Đề xuất project (W3)

### Bài toán
Xây dựng hệ thống tìm lộ trình tối ưu giữa hai ga bất kỳ trên mạng tram Manchester Metrolink (vùng Greater Manchester, Anh).

- **Đầu vào:** ga đi, ga đến, giờ khởi hành và tiêu chí tối ưu (nhanh nhất hoặc ít đổi tuyến nhất).
- **Đầu ra:** dãy ga đi qua, các điểm đổi tuyến, thời điểm đến từng ga và tổng thời gian, hiển thị trên bản đồ.

Bài toán được phát biểu dưới dạng tìm kiếm trên không gian trạng thái:

| Thành phần | Định nghĩa |
|---|---|
| Trạng thái | Cặp (ga, tuyến) mà hành khách đang ở |
| Trạng thái đầu | Mọi cặp (ga đi, tuyến), vì hành khách được lên bất kỳ tuyến nào tại ga đi |
| Hành động | Đi tiếp một ga trên tuyến hiện tại, hoặc đổi sang tuyến khác tại cùng ga |
| Chi phí | Thời gian chạy tàu giữa hai ga; 300 giây cho một lần đổi tuyến; cả hai tăng trong giờ cao điểm |
| Kiểm tra đích | Trạng thái thuộc ga đến, ở bất kỳ tuyến nào |

**Dữ liệu:** feed GTFS công khai của Transport for Greater Manchester (TfGM). Sau khi lọc riêng tram, mạng lưới gồm 7 tuyến, 99 ga, 192 trạng thái và 378 cạnh chạy tàu có hướng.

**Lý do chọn:** đây là bài toán tìm kiếm có thông tin trên dữ liệu thật. Tọa độ địa lý của ga cho phép xây dựng hàm heuristic, và chi phí thay đổi theo giờ trong ngày làm bài toán khó hơn tìm đường đi ngắn nhất trên đồ thị tĩnh.

### Phương pháp dự kiến
1. **Mô hình hóa đồ thị.** Từ dữ liệu GTFS, dựng đồ thị có hướng có trọng số. Mỗi nút là một cặp (ga, tuyến). Cạnh chạy tàu nối hai ga liên tiếp trên cùng tuyến, trọng số là trung vị thời gian chạy theo lịch. Cạnh đổi tuyến nối các nút cùng ga, trọng số 300 giây.
2. **Thuật toán A\*.** Hàm đánh giá $f(n) = g(n) + h(n)$, với $g(n)$ là thời gian đã đi và $h(n)$ là khoảng cách Haversine từ $n$ tới ga đích chia cho vận tốc tối đa của tàu. Heuristic này chấp nhận được vì không chuyến tàu nào chạy nhanh hơn vận tốc tối đa theo đường thẳng.
3. **Chi phí phụ thuộc thời gian.** Trong các khung giờ cao điểm, thời gian chạy tàu nhân với hệ số lưu lượng và mỗi lần đổi tuyến cộng thêm thời gian chờ. Hệ số được tra theo thời điểm hành khách tới nút, không phải giờ khởi hành.
4. **Tiêu chí ít đổi tuyến.** Dùng cùng thuật toán với hàm chi phí khác: mỗi lần đổi tuyến bị phạt rất nặng so với mỗi chặng tàu.
5. **Đánh giá.** So sánh A\* với Dijkstra (A\* có $h = 0$) trên toàn bộ các cặp ga về tính tối ưu và số nút mở rộng.
6. **Hệ thống.** Pipeline dữ liệu bằng Python (pandas) ghi ra SQLite; lõi tìm đường bằng C++; API bằng Node.js (Express) với MySQL; giao diện bản đồ bằng React và Leaflet.

### Phân công
- BH Nam: thuật toán A\* (lõi C++), thiết kế hàm heuristic, thực nghiệm đánh giá, tổng hợp báo cáo.
- NT Kiệt: thu thập và làm sạch dữ liệu GTFS, dựng đồ thị, cơ sở dữ liệu SQLite và MySQL.
- NH Nam: API Node.js, giao diện React và bản đồ Leaflet, kiểm thử toàn hệ thống.

## 3. Tiến độ giữa kỳ (W8)

### Chương trình
<!-- Bản nháp viết ngày 03/10/2026 theo kế hoạch và kết quả chạy thử pipeline. Cập nhật cột "Trạng thái" theo tiến độ thật của nhóm trước khi nộp ngày 24/10. -->
Đến giữa kỳ, nhóm hoàn thành phần dữ liệu và lõi thuật toán; phần web đang tích hợp.

| Thành phần | File | Chức năng | Trạng thái |
|---|---|---|---|
| Làm sạch dữ liệu | `data_pipeline/data_cleaner.ipynb` | Lọc tram khỏi feed GTFS, gộp sân ga thành ga, tạo nút (ga, tuyến) | Hoàn thành |
| Dựng đồ thị | `data_pipeline/database_builder.ipynb` | Tính cạnh và thời gian chạy, ghi SQLite và file SQL cho MySQL | Hoàn thành |
| Lõi tìm đường | `backend/main.cpp` | A\* phụ thuộc thời gian, hai tiêu chí, trả kết quả JSON | Hoàn thành, chạy từ dòng lệnh |
| API | `backend/server.js` | Gọi lõi C++, đăng nhập, quản lý giờ cao điểm | Đang tích hợp |
| Giao diện | `frontend/` | Nhập ga, hiển thị lộ trình trên bản đồ, trang quản trị | Đang tích hợp |

Lõi tìm đường chạy độc lập bằng lệnh `metro.exe <nút đi> <nút đến> <chế độ> <giờ>`. Ví dụ `metro.exe 3 27 1 12:00` tìm đường nhanh nhất từ Altrincham đến Bury lúc 12:00 và in ra danh sách ga kèm thời điểm đến dưới dạng JSON.

### Phân tích kết quả, vấn đề gặp phải
**Kết quả xử lý dữ liệu** (feed TfGM tải ngày 03/10/2026):

| Đại lượng | Toàn feed | Sau khi lọc tram |
|---|---|---|
| Tuyến | 656 | 7 tuyến (8 `route_id`) |
| Chuyến | 69.680 | 14.745 |
| Dòng `stop_times` | 2.902.195 | 273.290 |
| Điểm dừng | 15.693 | 199 sân ga thuộc 99 ga |

Đồ thị thu được có 192 nút (ga, tuyến), 378 cạnh chạy tàu có hướng và 390 cạnh đổi tuyến tại 44 ga trung chuyển. Thời gian chạy giữa hai ga liên tiếp từ 60 đến 360 giây, trung bình 158 giây.

**Vấn đề gặp phải và cách xử lý:**

1. **Xe buýt thay thế lẫn trong tuyến tram.** Feed có 12 tuyến mang `route_type = 0`, trong đó 4 tuyến là "Replacement bus". Nhóm lọc thêm theo tên tuyến.
2. **Điểm dừng là sân ga, không phải ga.** Mỗi `stop_id` là một sân ga theo chiều và nhiều tuyến dùng chung sân ga, nên không thể suy ra việc đổi tuyến từ `stop_id`. Nhóm gộp sân ga thành ga theo 11 ký tự đầu của `stop_code`, rồi tạo nút theo cặp (ga, tuyến).
3. **Giờ vượt quá 24:00.** GTFS ghi giờ tới `26:12:30` cho chuyến qua nửa đêm (10.324 dòng, 442 cặp ga liên tiếp vắt qua nửa đêm). Quy về 0–24 giờ sẽ sinh thời gian chạy âm, nên nhóm giữ nguyên và tính hiệu trực tiếp.
4. **Thời gian chạy không đồng nhất.** Cùng một cạnh có nhiều giá trị giữa các chuyến, và có 3 dòng thời gian bằng 0 trong 258.545 dòng. Nhóm lấy trung vị cho mỗi cạnh.
5. **Phụ thuộc vào tuyến của ga xuất phát.** Nếu chỉ xuất phát từ một nút (ga, tuyến) cố định, lộ trình Altrincham → Bury mất 67 phút với một lần đổi tuyến thừa. Khi cho mọi nút của ga đi cùng là trạng thái đầu, kết quả là 62 phút đi thẳng.

**Kết quả chạy thử lõi tìm đường** (tiêu chí nhanh nhất, khởi hành 12:00):

| Ga đi | Ga đến | Thời gian | Số ga | Đổi tuyến |
|---|---|---|---|---|
| Altrincham | Bury | 62,0 phút | 25 | Không |
| St Peter's Square | Manchester Airport | 51,5 phút | 22 | Không |
| Eccles | Ashton-under-Lyne | 68,0 phút | 27 | Không |
| The Trafford Centre | East Didsbury | 42,0 phút | 17 | Tại Trafford Bar |
| Manchester Airport | Rochdale Town Centre | 117,0 phút | 44 | Tại Victoria |

## 4. Cập nhật kết quả cuối kỳ (W15)

### Chi tiết phương pháp, dữ liệu
<!-- Bản nháp viết ngày 03/10/2026. Số liệu lấy từ lần chạy thử trên feed tải ngày 03/10/2026; chạy lại data pipeline và evaluate.py rồi cập nhật trước khi nộp ngày 12/12. -->
**Dữ liệu.** Nguồn là feed GTFS "GM Public Transport Schedules" của TfGM (`https://odata.tfgm.com/opendata/downloads/TfGMgtfsnew.zip`), cập nhật hằng đêm, giấy phép ODbL v1.0 và OGL v3.0 (Contains Transport for Greater Manchester data). Nhóm dùng 4 file: `routes.txt`, `trips.txt`, `stop_times.txt` và `stops.txt`. Tên tuyến lấy theo `route_short_name` trong feed.

**Đồ thị.** $G = (V, E)$ có hướng, với $|V| = 192$ nút (ga, tuyến) thuộc 99 ga và 7 tuyến.

- *Cạnh chạy tàu* (378 cạnh): nối hai ga liên tiếp của cùng một tuyến. Trọng số $w(e)$ là trung vị của hiệu giờ đến ga sau và giờ rời ga trước trên mọi chuyến đi qua cạnh đó, nằm trong khoảng 60–360 giây.
- *Cạnh đổi tuyến* (390 cạnh): nối mọi cặp nút cùng ga, trọng số $w(e) = 300$ giây, biểu diễn thời gian chờ chuyến của tuyến khác.

**Chi phí phụ thuộc thời gian.** Khi đi qua cạnh $e$ tại thời điểm $t$ trong ngày:

$$c(e, t) = w(e)\,\alpha(t) + \mathbb{1}[e \text{ là đổi tuyến}]\,\beta(t)$$

trong đó $\alpha(t)$ là hệ số lưu lượng và $\beta(t)$ là thời gian chờ thêm của khung giờ chứa $t$; ngoài giờ cao điểm $\alpha = 1$, $\beta = 0$. Quản trị viên chỉnh các khung giờ này qua giao diện. Giá trị mặc định là giả định của nhóm, vì TfGM không công bố dữ liệu tương ứng:

| Khung giờ | $\alpha$ | $\beta$ |
|---|---|---|
| 07:00–09:30 | 1,5 | 300 giây |
| 16:00–18:30 | 1,3 | 180 giây |

**Thuật toán A\*.** $f(n) = g(n) + h(n)$, trong đó $g(n)$ là tổng chi phí từ ga đi và

$$h(n) = \frac{d_{\text{Haversine}}(n, \text{đích})}{v_{\max}}, \qquad v_{\max} = 20 \text{ m/s}$$

- *Trạng thái đầu:* mọi nút thuộc ga đi được đưa vào hàng đợi với $g = 0$.
- *Kiểm tra đích:* dừng khi lấy ra khỏi hàng đợi một nút có tên ga trùng ga đến.
- *Tính chấp nhận được:* vận tốc đường thẳng lớn nhất trên mọi cạnh là 14,12 m/s (Whitefield → Radcliffe), nhỏ hơn $v_{\max}$, nên $h(u) - h(v) \le w(u, v)$ với mọi cạnh. Cạnh đổi tuyến nối hai nút cùng tọa độ nên không làm thay đổi $h$. Vì $\alpha \ge 1$ và $\beta \ge 0$, giờ cao điểm chỉ làm chi phí thật tăng. Do đó $h$ nhất quán, và A\* trả về lời giải tối ưu khi chi phí không đổi theo thời gian; trường hợp phụ thuộc thời gian được kiểm chứng bằng thực nghiệm ở phần đánh giá.

**Tiêu chí ít đổi tuyến.** Dùng cùng thuật toán với chi phí $\alpha(t)$ cho mỗi cạnh chạy tàu và $10\,000 + \beta(t)$ cho mỗi cạnh đổi tuyến, nên số lần đổi tuyến được ưu tiên tuyệt đối so với số chặng.

### Chương trình
**Kiến trúc:**

```text
Feed GTFS (TfGM)
   │  data_cleaner.ipynb, database_builder.ipynb (Python, pandas)
   ▼
metrolink.db (SQLite) ──► metro.exe (C++, A*) ──┐
database.sql ──► MySQL                           │ JSON
                   ▲                             ▼
                   └──── server.js (Node.js, Express) ◄──► React + Leaflet
```

| Tầng | Công nghệ | Vai trò |
|---|---|---|
| Dữ liệu | Python, pandas | Làm sạch GTFS, dựng đồ thị |
| Lưu trữ | SQLite, MySQL | SQLite cho lõi C++ đọc; MySQL cho tài khoản, danh sách ga và giờ cao điểm |
| Thuật toán | C++ | A\* phụ thuộc thời gian, công thức Haversine |
| API | Node.js, Express, JWT, bcrypt | Gọi lõi C++, xác thực, đồng bộ hai cơ sở dữ liệu |
| Giao diện | React, Vite, Tailwind CSS, React-Leaflet | Tìm đường, bản đồ, trang quản trị |

**Chức năng cho người dùng:** chọn ga đi và ga đến có gợi ý tên, chọn giờ khởi hành và tiêu chí, xem lộ trình từng ga với thời điểm đến và điểm đổi tuyến, xem đường đi trên bản đồ.

**Chức năng cho quản trị viên:** đăng nhập phân quyền, thêm hoặc xóa khung giờ cao điểm với hệ số lưu lượng và thời gian chờ, đóng hoặc mở ga. Thay đổi có hiệu lực ngay ở lần tìm đường kế tiếp.

**API:**

| Phương thức | Đường dẫn | Chức năng |
|---|---|---|
| GET | `/api/stations` | Danh sách nút ga |
| GET | `/api/routing?start=&end=&mode=&time=` | Tìm đường bằng lõi C++ |
| POST | `/api/admin/station/status` | Đóng hoặc mở ga |
| GET, POST, DELETE | `/api/admin/peak-hours` | Quản lý khung giờ cao điểm |
| POST | `/api/register`, `/api/login` | Đăng ký, đăng nhập |

**Cách chạy:** chạy hai notebook trong `data_pipeline/` để sinh `metrolink.db` và `database.sql`; import `database.sql` vào MySQL; biên dịch `main.cpp` thành `metro.exe`; chạy `node server.js` trong `backend/` và `npm run dev` trong `frontend/`; mở `http://localhost:5173`.

### Phân tích, đánh giá kết quả
Thực nghiệm chạy trên toàn bộ $99 \times 98 = 9\,702$ cặp ga có thứ tự, với feed TfGM tải ngày 03/10/2026. Script `backend/evaluate.py` mô phỏng lại logic của lõi C++ để đếm số nút mở rộng và so sánh A\* với Dijkstra (A\* có $h = 0$).

**1. Tính tối ưu.** A\* cho tổng chi phí trùng với Dijkstra ở 9.702/9.702 cặp, cả khi khởi hành 12:00 (thấp điểm) lẫn 08:00 (cao điểm).

**2. Hiệu quả tìm kiếm.** Số nút mở rộng trung bình mỗi truy vấn:

| Giờ khởi hành | $v_{\max}$ của heuristic | A\* | Dijkstra | A\* giảm |
|---|---|---|---|---|
| 12:00 | 20 m/s | 91,6 | 108,4 | 15,6% |
| 08:00 | 20 m/s | 95,3 | 107,2 | 11,1% |
| 12:00 | 15 m/s | 85,7 | 108,4 | 20,9% |
| 08:00 | 15 m/s | 91,2 | 107,2 | 14,9% |

A\* mở rộng ít nút hơn Dijkstra nhưng mức giảm khiêm tốn, vì ba lý do. Thứ nhất, $v_{\max} = 20$ m/s cao hơn nhiều so với vận tốc thật của tram, nên $h$ đánh giá thấp chi phí còn lại. Thứ hai, đồ thị nhỏ và gồm các nhánh hướng tâm, nên phần lớn nút nằm gần đường đi tối ưu. Thứ ba, trong giờ cao điểm chi phí thật tăng còn $h$ giữ nguyên, nên heuristic càng yếu. Hạ $v_{\max}$ xuống 15 m/s (vẫn lớn hơn 14,12 m/s nên vẫn chấp nhận được) tăng mức giảm lên 20,9%.

**3. Thời gian chạy.** Mỗi lần gọi `metro.exe` mất trung bình 17–20 ms (đo 20 lần cho mỗi truy vấn trong 4 truy vấn), gồm cả khởi tạo tiến trình, đọc SQLite và dựng đồ thị. Thời gian tìm kiếm trên 192 nút không đáng kể so với các bước đó.

**4. Chất lượng lộ trình.**

| Đại lượng | Khởi hành 12:00 | Khởi hành 08:00 |
|---|---|---|
| Thời gian hành trình trung bình | 41,1 phút | 63,2 phút |
| Trung vị | 40,0 phút | 63,0 phút |
| Dài nhất (Manchester Airport → Rochdale Town Centre) | 117,0 phút | 150,5 phút |

Có 3.876 cặp ga (40%) đi thẳng không đổi tuyến và 5.826 cặp (60%) cần đúng một lần đổi tuyến; không cặp nào cần hai lần. Một số lộ trình tiêu biểu:

| Ga đi | Ga đến | 12:00 | 08:00 | Đổi tuyến |
|---|---|---|---|---|
| Altrincham | Bury | 62,0 phút | 93,0 phút | Không |
| The Trafford Centre | East Didsbury | 42,0 phút | 68,0 phút | Tại Trafford Bar |
| Manchester Airport | Rochdale Town Centre | 117,0 phút | 150,5 phút | Tại Victoria (12:00), St Peter's Square (08:00) |

Lộ trình Manchester Airport → Rochdale Town Centre cho thấy tác dụng của chi phí phụ thuộc thời gian: giờ cao điểm kết thúc lúc 09:30 khi hành trình còn dang dở, nên tổng thời gian chỉ tăng 29% thay vì 50%, và điểm đổi tuyến chuyển từ Victoria sang St Peter's Square.

**5. Hạn chế.**

- Thời gian chờ tàu là hằng số 300 giây, chưa lấy từ tần suất chuyến thật; hệ số giờ cao điểm là giả định.
- Ở tiêu chí ít đổi tuyến, chi phí tính theo số chặng nhưng $h$ vẫn tính theo giây, nên $h$ không còn chấp nhận được. Số lần đổi tuyến vẫn tối thiểu ở cả 9.702 cặp (mức phạt 10.000 lớn hơn giá trị lớn nhất của $h$ là 1.454), nhưng 605 cặp (6%) có số chặng nhiều hơn mức tối thiểu, nhiều nhất 4 chặng.
- Hệ số giờ cao điểm thay đổi đột ngột tại biên khung giờ, nên mạng không hoàn toàn thỏa tính FIFO. Phép so sánh với Dijkstra không phát hiện được sai lệch loại này, vì cả hai cùng giả định hành khách không chờ ở ga.
- Thao tác đóng ga áp dụng cho từng nút (ga, tuyến), nên muốn đóng hẳn một ga phải đóng mọi nút của ga đó.
- API quản trị chưa kiểm tra token ở phía server; phân quyền mới thực hiện ở giao diện.

**6. Hướng phát triển.** Tính thời gian chờ theo tần suất chuyến trong `stop_times.txt`; dùng $h = 0$ hoặc heuristic theo số chặng cho tiêu chí ít đổi tuyến; hạ $v_{\max}$ xuống 15 m/s; thêm kiểm tra token cho API quản trị.

### Cập nhật phân công, khối lượng công việc
<!-- công việc của các thành viên, tỷ lệ đóng góp của các thành viên -->
<!-- Phân công dưới đây là đề xuất. Nhóm sửa lại theo thực tế và điền cột "Tỷ lệ đóng góp" trước khi nộp. -->

| Thành viên | Công việc | Tỷ lệ đóng góp |
|---|---|---|
| Bùi Hải Nam (20235384) | Lõi A\* bằng C++ (trạng thái đầu đa nút, chi phí phụ thuộc thời gian, hai tiêu chí); phân tích tính chấp nhận được của heuristic; thực nghiệm so sánh với Dijkstra; tổng hợp báo cáo | ...% |
| Nguyễn Tuấn Kiệt (20235357) | Khảo sát feed GTFS của TfGM; notebook làm sạch dữ liệu và dựng đồ thị; cơ sở dữ liệu SQLite và MySQL | ...% |
| Nguyễn Hà Nam (20235174) | API Node.js và đồng bộ hai cơ sở dữ liệu; giao diện React, bản đồ Leaflet, trang quản trị; kiểm thử toàn hệ thống | ...% |